In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_performance_comercial AS
SELECT
  TO_DATE(p.order_date)                                                           AS order_date,
  p.status_order,
  v.canal_id,
  v.regional_code,
  pr.category,
  pr.name,
  SUM(p.gross_amount_num)                                                         AS receita_bruta,
  SUM(p.net_amount)                                                               AS receita_liquida,
  SUM(p.discount_amount)                                                          AS desconto_total,
  ROUND(
    SUM(p.discount_amount) * 100.0
    / NULLIF(SUM(p.gross_amount_num), 0), 2
  )                                                                               AS perc_desconto,
  AVG(p.gross_amount_num)                                                         AS ticket_medio,
  SUM(i.quantity)                                                                 AS total_itens
FROM workspace.silver.tb_pedidos_cabecalho p
LEFT JOIN workspace.silver.dim_vendedores   v  ON p.seller_id    = v.seller_id
LEFT JOIN workspace.silver.tb_pedidos_itens i  ON p.order_id     = i.order_id
LEFT JOIN workspace.silver.dim_produto      pr ON i.product_code = pr.product_id
GROUP BY
  TO_DATE(p.order_date),
  p.status_order,
  v.canal_id,
  v.regional_code,
  pr.category,
  pr.name